# Flower tracking: perception and uncertainty reduction

Run in order to train the recurrent perception model, inspect action-space UR
maps, and compare tracking strategies and light conditions. Training uses
shuffled moth trajectories.

In [ ]:
from pathlib import Path
import os
import sys

PACKAGE_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "NectarySearch").is_dir() and (p / "FlowerTracking").is_dir()),
    None,
)
if PACKAGE_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted code folder.")
NOTEBOOK_DIR = PACKAGE_ROOT / "FlowerTracking"
sys.path.insert(0, str(PACKAGE_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))
os.chdir(NOTEBOOK_DIR)

import math
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import scienceplots
from task import (
    MothDataset, ShuffledMothDataset, DrunkMothDataset, DistractorMothDataset,
    ImageMothDataset, MothDatasetByCondition, compute_visual_obs,
)
from observation_models import generate_proboscis_observations_1d
from training import train_rnn_model
from model import RNNModel
from utils.plotting import plot_raincloud, plot_confidence_intervals

plt.rcParams['figure.figsize'] = (20, 5)

In [ ]:
def set_seed(seed):
    """Sets the random seed for reproducibility across multiple libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [ ]:
set_seed(657)

In [ ]:
def compute_entropy(logvar_z):
    return 0.5 * (1 + math.log(2 * torch.pi)) + 0.5 * logvar_z

## Data and observation settings

In [ ]:
data_dir = "data"

In [ ]:
nu = 0.06
nu_distract = 0.01
sigma = 0.06
A = nu * np.sqrt(2 * np.pi)
A_distract = 0.02
mech_params = {
    'tau_p': 2e-2,
    'd_sat': 8e-2,
    'L_prob': 0.1,
    'sigma_frac': 0.06,
    'F_thr': 0.005
}
crop_sz = (12, 12)

In [ ]:
dataset = MothDataset(data_dir, nu=nu, sigma=sigma, normalize=True, seq_len=200, A=A, mechanosensory=True, mech_obs_params=mech_params)

shuffled_dataset = ShuffledMothDataset(data_dir, nu=nu, sigma=sigma, normalize=True, seq_len=200, A=A, mechanosensory=True, mech_obs_params=mech_params)

drunk_moth_dataset = DrunkMothDataset(data_dir, nu=nu, sigma=sigma, normalize=True, seq_len=200, A=A, mechanosensory=False, mech_obs_params=mech_params)

distractor_dataset = DistractorMothDataset(data_dir, nu=nu, sigma=sigma, normalize=True, seq_len=200, A=A,
                                           dist_A=A_distract, dist_nu=nu_distract, dist_sigma=0.1)

mech_dataset = MothDataset(data_dir, nu=nu, sigma=sigma, normalize=True, seq_len=200, A=A,
                           mechanosensory=True, mech_obs_params=mech_params)

image_dataset = ImageMothDataset(drunk_moth_dataset, world_size_px=(48, 48), crop_size_px=crop_sz,
                                 flower_radius_px=5, grayscale=True)

## Example trajectories and observations

In [ ]:
randindx = np.random.randint(0, high=len(dataset))
st1, at1, xt1 = dataset.__getitem__(randindx)
st2, at2, xt2 = shuffled_dataset.__getitem__(randindx)
st3, at3, xt3 = drunk_moth_dataset.__getitem__(randindx)
(st4, distractor_s), at4, xt4 = distractor_dataset.__getitem__(randindx)

with plt.style.context(['ieee', 'notebook', 'no-latex']):

    plt.figure(figsize=(10, 6))
    fig, axs_ = plt.subplots(2, 1, sharex=True)
    ax1, ax2 = axs_[0], axs_[1]
    ax1.plot(st1.numpy(), label='flower')
    ax1.plot(at1.numpy(), label='moth')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.set_ylim([0, 1])
    ax1.legend(frameon=False)
    ax2.plot(xt1.numpy()[:,0], '-', color='purple')
    ax2.plot(xt1.numpy()[:,1], '-', color='darkorange')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.legend(['Visual', 'Mechanosensory'], frameon=False)

    plt.xlabel('Timestep')

    plt.figure(figsize=(15, 6))
    ax1 = plt.subplot(2, 1, 1)
    ax1.plot(st2.numpy(), label='flower location')
    ax1.plot(at2.numpy(), label='moth location')
    plt.ylim([0, 1])
    plt.legend()
    ax2 = plt.subplot(2, 1, 2)
    ax2.plot(xt2.numpy(), label='sensory observation')
    plt.legend()

    plt.xlabel('Timestep')

    plt.figure(figsize=(15, 6))
    ax1 = plt.subplot(2, 1, 1)
    ax1.plot(st3.numpy(), label='flower location')
    ax1.plot(at3.numpy(), label='moth location')
    plt.ylim([0, 1.05])

    plt.legend()
    ax2 = plt.subplot(2, 1, 2)
    ax2.plot(xt3.numpy(), label='sensory observation')
    plt.legend()

    plt.xlabel('Timestep')

    plt.figure(figsize=(15, 6))
    ax1 = plt.subplot(2, 1, 1)
    ax1.plot(st4.numpy(), label='flower location')
    ax1.plot(at4.numpy(), label='moth location')
    ax1.plot(distractor_s.numpy(), label='distractor location')
    plt.ylim([0, 1.05])
    plt.legend(loc='upper left')
    ax2 = plt.subplot(2, 1, 2)
    ax2.plot(xt4.numpy(), label='sensory observation')
    plt.legend()

    plt.xlabel('Timestep')

    plt.show()

## Train the perception model

In [ ]:
training_dataset = shuffled_dataset
model = RNNModel(obs_dim=training_dataset.x.shape[-1], action_dim=1, latent_dim=1, hidden_dim=16, beta=0.2)
_ = train_rnn_model(num_epochs=100, batch_size=100, dataset=training_dataset, model=model, synthetic_action=False,
                    train_action_net=False, ur_weight=1.0,
                    shuffle_every_epoch=True)

## Reconstructions and posterior uncertainty

In [ ]:
model = model.cpu()
with torch.no_grad():
    s, a, x = dataset.__getitem__(np.random.randint(0, len(dataset)))
    s, a, x = s.unsqueeze(0), a.unsqueeze(0), x.unsqueeze(0)

    r, losses = model(x, a)

In [ ]:
with plt.style.context(['nature', 'notebook', 'no-latex']):
    plt.plot(s.cpu().detach().numpy().squeeze(), label='Flower Location')
    plt.plot(a.cpu().detach().numpy().squeeze(), label='Moth Location')

    plt.legend()
    plt.show()

In [ ]:
with plt.style.context(['nature', 'notebook', 'no-latex']):
    plt.plot(x.cpu().detach().numpy().squeeze(), label='Actual')
    plt.plot(r['x_hat'].cpu().detach().numpy().squeeze(), label='Reconstructed')
    plt.legend()
    plt.show()

In [ ]:
with plt.style.context(['nature', 'notebook', 'no-latex']):

    plt.plot(r['post_mu'].squeeze().cpu().detach().numpy().squeeze(), label='Posterior Mean')

    plt.plot(r['prior_mu'].squeeze().cpu().detach().numpy().squeeze(), label='Prior Mean')

    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
prior_entropy = compute_entropy(r['prior_logvar'])
post_entropy = compute_entropy(r['post_logvar'])
entropy = prior_entropy - post_entropy

In [ ]:
plt.plot(entropy.sum(-1, keepdim=True).cpu().detach().numpy().squeeze(), label='entropy')

plt.legend()
plt.show()

## Entropy reduction over candidate positions

The map evaluates 100 candidate positions for 100 example trajectories. Each
column is rescaled across candidate positions for display.

In [ ]:
with_distractor = False
entmap_dataset = dataset

range_sz = 100
s_samples = 100
action_range = torch.linspace(0., 1., range_sz).reshape(-1, 1, 1)
action_range = action_range.repeat_interleave(dim=1, repeats=s_samples)
action_range = action_range.repeat_interleave(dim=-1, repeats=entmap_dataset.a.shape[-1])

if not with_distractor:
    s_range = entmap_dataset.s[:s_samples].unsqueeze(0).repeat_interleave(dim=0, repeats=range_sz)

    if not isinstance(entmap_dataset, ImageMothDataset):
        x_range = compute_visual_obs(s_range, action_range, sigma, nu, A)
        if entmap_dataset.mechanosensory:
            mech_x = generate_proboscis_observations_1d(s_range.cpu().numpy().reshape(-1, s_range.shape[-1]),
                                                        action_range.cpu().numpy().reshape(-1, action_range.shape[-1]), dt=1e-2, **mech_params)
            mech_x = torch.tensor(mech_x).to(x_range).reshape(*x_range.shape)
            x_range = torch.stack([x_range, mech_x], dim=-1)
    else:
        flower_xy = [[np.stack([si.numpy(), 0.5 * np.ones(si.shape)], axis=-1) for si in sr] for sr in s_range]
        moth_xy = [[np.stack([ai.numpy(), 0.5 * np.ones(ai.shape)], axis=-1) for ai in ar] for ar in action_range]
        x_range_vis = [[entmap_dataset.renderer.render_frames(fxy, mxy) for fxy, mxy in zip(fr, mr)] for fr, mr in zip(flower_xy, moth_xy)]
        x_range_vis = torch.Tensor(np.array(x_range_vis)).flatten(start_dim=-2)

        if entmap_dataset.mechanosensory:
            x_range_mech = generate_proboscis_observations_1d(s_range.cpu().numpy().reshape(-1, s_range.shape[-1]),
                                                              action_range.cpu().numpy().reshape(-1, action_range.shape[-1]), dt=1e-2, **mech_params)
            x_range_mech = torch.tensor(x_range_mech).to(x_range_vis).reshape(*s_range.shape).unsqueeze(-1)
            x_range = torch.concatenate([x_range_vis, x_range_mech], dim=-1)
        else:
            x_range = x_range_vis
else:
    s_range = distractor_dataset.s.unsqueeze(1).repeat_interleave(dim=1, repeats=range_sz)
    dist_s_range = distractor_dataset.distractor_s.unsqueeze(1).repeat_interleave(dim=1, repeats=range_sz)
    x_range = compute_visual_obs(s_range, action_range, sigma, nu, A) + compute_visual_obs(dist_s_range, action_range, sigma, nu_distract, A_distract)

In [ ]:
model.enum_action = None
with torch.no_grad():
    r_moth_traj = model(entmap_dataset.x, entmap_dataset.a)[0]

In [ ]:
with torch.no_grad():
    ht = r_moth_traj['h'][:s_samples].unsqueeze(0).repeat_interleave(dim=0, repeats=range_sz).flatten(0, 2)

    m, lv = model.cpu().prior_t(ht, action_range.reshape(-1, 1))
    m, lv = m.reshape(range_sz, s_samples, 200, model.latent_dim), lv.reshape(range_sz, s_samples, 200, model.latent_dim)

    mp, lvp = model.encode_t(x_range.reshape(-1, entmap_dataset.x.shape[-1]), action_range.reshape(-1, 1), ht)
    mp, lvp = mp.reshape(range_sz, s_samples, 200, model.latent_dim), lvp.reshape(range_sz, s_samples, 200, model.latent_dim)

    ent_range = (compute_entropy(lv) - compute_entropy(lvp)).squeeze().detach()
    ent_range = ent_range.reshape(range_sz, s_samples, dataset.s.shape[-1])
    ent_range = ent_range.permute(1, 0, 2)

In [ ]:
ent_range = (ent_range - ent_range.min(1, keepdim=True)[0]) / (ent_range.max(1, keepdim=True)[0] - ent_range.min(1, keepdim=True)[0])

In [ ]:
exidx = np.random.randint(ent_range.shape[0])

include_track_error = False

with plt.style.context(['nature', 'notebook', 'no-latex']):
    plt.figure(figsize=(8, 5))

    to_show = ent_range[exidx].numpy()

    if include_track_error:
        to_show -= (dataset.s[exidx] - action_range[exidx]).abs().numpy()

    plt.imshow(to_show, aspect='auto', origin='lower', cmap='viridis', interpolation='nearest', extent=[0, 200, 0, 1])
    plt.plot(dataset.s[exidx].numpy(), 'r-', linewidth=3., label='Flower')
    plt.plot(dataset.a[exidx].numpy(), 'k--', linewidth=4., label='Moth')

    if with_distractor:
        plt.plot(distractor_dataset.distractor_s[exidx].numpy(), '-', color='darkorange', linewidth=3., label='Distractor')
    plt.colorbar(label='$\Delta H_t$')
    plt.legend(bbox_to_anchor=(1.2, 1), loc='upper left')
    plt.ylabel("$a_{t-1}$")
    plt.xlabel("Time step $(t)$")
    plt.show()

## Entropy reduction for tracking strategies

Compare actual moth trajectories, exact tracking of the flower, and shuffled
moth trajectories.

In [ ]:
true_dl = DataLoader(dataset, batch_size=len(dataset), shuffle=False)

true_s, true_a, true_x = next(iter(true_dl))

In [ ]:
num_shuff = 20
shuff_datasets = [ShuffledMothDataset(data_dir, nu=nu, sigma=sigma, normalize=True, seq_len=200, A=A, mechanosensory=True, mech_obs_params=mech_params) for _ in range(num_shuff)]

shuffled_s, shuffled_a, shuffled_x = [], [], []
for sd in shuff_datasets:
    si, ai, xi = next(iter(DataLoader(sd, batch_size=len(sd), shuffle=False)))
    shuffled_s += [si]
    shuffled_a += [ai]
    shuffled_x += [xi]
shuffled_s = torch.stack(shuffled_s, dim=0)
shuffled_a = torch.stack(shuffled_a, dim=0)
shuffled_x = torch.stack(shuffled_x, dim=0)

In [ ]:
noisy_a = true_a + torch.randn_like(true_a) * 0.05
noisy_x = compute_visual_obs(true_s, noisy_a, sigma, nu, A)

if dataset.mechanosensory:
    mech_x = generate_proboscis_observations_1d(true_s.cpu().numpy(), noisy_a.cpu().numpy(),
                                                dt=1e-2, **mech_params)
    mech_x = torch.tensor(mech_x).to(noisy_x)
    noisy_x = torch.stack([noisy_x, mech_x], dim=-1)

In [ ]:
noisy_f = true_s
noisy_fx = compute_visual_obs(true_s, noisy_f, sigma, nu, A)
if dataset.mechanosensory:
    mech_x = generate_proboscis_observations_1d(true_s.cpu().numpy(), noisy_f.cpu().numpy(),
                                                dt=1e-2, **mech_params)
    mech_x = torch.tensor(mech_x).to(noisy_x)
    noisy_fx = torch.stack([noisy_fx, mech_x], dim=-1)

In [ ]:
exi = 4
plt.subplot(1, 2, 1)
plt.plot(true_s[exi].numpy(), label='True flower')
plt.plot(true_a[exi].numpy(), label='true moth')
plt.plot(shuffled_a[0, exi].numpy(), label='shuffled moth')
plt.plot(noisy_a[exi].numpy(), label='noisy moth')
plt.plot(noisy_f[exi].numpy(), label='exact flower')
plt.legend()
plt.title("True vs. shuffled vs. noisy locations")
plt.subplot(1, 2, 2)
plt.plot(true_x[exi].numpy(), label='true')
plt.plot(shuffled_x[0, exi].numpy(), label='shuffled')
plt.plot(noisy_x[exi].numpy(), label='noisy moth')
plt.plot(noisy_fx[exi].numpy(), label='exact flower')
plt.legend()
plt.title("True vs. noisy vs. randomized observations")
plt.show()

In [ ]:
with torch.no_grad():
    model = model.cpu()
    shuffled_r, _ = model(shuffled_x.flatten(0, 1), shuffled_a.flatten(0, 1))
    shuffled_r['prior_logvar'] = shuffled_r['prior_logvar'].reshape(num_shuff, shuffled_s.shape[1], -1)
    shuffled_r['post_logvar'] = shuffled_r['post_logvar'].reshape(num_shuff, shuffled_s.shape[1], -1)

In [ ]:
with torch.no_grad():
    true_r, _ = model(true_x, true_a)
    noisy_r, _ = model(noisy_x, noisy_a)
    noisyf_r, _ = model(noisy_fx, noisy_f)

In [ ]:
true_ent_red = (compute_entropy(true_r['prior_logvar']) - compute_entropy(true_r['post_logvar'])).squeeze().detach()
shuffled_ent_red = (compute_entropy(shuffled_r['prior_logvar']) - compute_entropy(shuffled_r['post_logvar'])).mean(0).squeeze().detach()
noisy_ent_red = (compute_entropy(noisy_r['prior_logvar']) - compute_entropy(noisy_r['post_logvar'])).squeeze().detach()
noisyf_ent_red = (compute_entropy(noisyf_r['prior_logvar']) - compute_entropy(noisyf_r['post_logvar'])).squeeze().detach()

In [ ]:
optimal_ent_red = torch.concatenate([true_ent_red, shuffled_ent_red, noisy_ent_red, noisyf_ent_red], dim=0).flatten().max(0)[0]
min_ent_red = torch.concatenate([true_ent_red, shuffled_ent_red, noisy_ent_red, noisyf_ent_red], dim=0).flatten().min(0)[0]

true_ent_red = ((true_ent_red - min_ent_red ) / optimal_ent_red) * 100
shuffled_ent_red = ((shuffled_ent_red - min_ent_red )/ optimal_ent_red) * 100
noisy_ent_red = ((noisy_ent_red - min_ent_red) / optimal_ent_red) * 100
noisyf_ent_red = ((noisyf_ent_red - min_ent_red) / optimal_ent_red) * 100

In [ ]:
with plt.style.context(['ieee', 'notebook', 'no-latex']):
    fig, ax = plot_raincloud(true_ent_red.squeeze().mean(-1).cpu().numpy(),
                   noisyf_ent_red.squeeze().mean(-1).cpu().numpy(),
                   shuffled_ent_red.squeeze().mean(-1).cpu().numpy(),

                labels=["Actual\nMoth", "Exact\nFlower", "Shuffled\nMoth"],
                y_label='Normalized mean UR',
                ylim=[0, 100], figsz=(5, 3))

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.show()

In [ ]:
with plt.style.context(['ieee', 'notebook', 'no-latex']):
    fig, ax = plot_confidence_intervals(true_ent_red.squeeze().mean(-1).cpu().numpy(),

                noisyf_ent_red.squeeze().mean(-1).cpu().numpy(),
                shuffled_ent_red.squeeze().mean(-1).cpu().numpy(),

                labels=["Actual Moth", "Exact Flower", "Shuffled Moth"],
                y_label='Normalized mean UR',

                ylim=[40, 65], alpha=0.01)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.show()

## Species and light conditions

The three light levels are 0.3, 15, and 300 lux. The corresponding observation
noise standard deviations are 0.5, 0.1, and 0.01.

In [ ]:
files = {
    "low": {
        "D. elpenor": ['Tracks_D0p3lux.pt']
    },
    "mid": {
        "D. elpenor": ['Tracks_D15lux.pt'],
        "M. stellatarum": ['Tracks_M15lux.pt']
    },
    "high": {
        "D. elpenor": ['Tracks_D300lux.pt'],
        "M. stellatarum": ['Tracks_M300lux.pt']
    }
}

obs_noises = {
    "low": 0.5,
    "mid": 0.1,
    "high": 0.01
}

cond_dataset = MothDatasetByCondition(data_dir, nu=nu, sigma=sigma, normalize=True, seq_len=200, A=A,
                                      condition_file_names=files, condition_sigmas=obs_noises,
                                      mechanosensory=True, mech_obs_params=mech_params)

_ = model.cpu()

In [ ]:
with torch.no_grad():
    cond_resps = {k1: {k2: model(cond_dataset.x[k1][k2], cond_dataset.a[k1][k2])[0] for k2 in files[k1].keys()} for k1 in files.keys()}
    cond_ers = {k1: {
        k2: (compute_entropy(cond_resps[k1][k2]['prior_logvar']) - compute_entropy(cond_resps[k1][k2]['post_logvar'])).squeeze().detach()
        for k2 in files[k1].keys()} for k1 in files.keys()}

In [ ]:
max_cond_ers = {k1: torch.concat([cond_ers[k1][k2] for k2 in files[k1]], dim=0).max(dim=0, keepdim=True)[0] for k1 in files.keys()}

In [ ]:
cond_ers = {
    k1: {
        k2: (cond_ers[k1][k2] / max_cond_ers[k1]) * 100 for k2 in files[k1].keys()
    } for k1 in files.keys()
}

In [ ]:
with plt.style.context(['ieee', 'notebook', 'no-latex']):
    for k1 in files.keys():

        ers = [cond_ers[k1][k2].mean(-1).cpu().numpy() for k2 in files[k1].keys()]

        fig, ax = plot_raincloud(*ers, labels=list(files[k1].keys()),
                    y_label='Avg. UR (% of max.)',

                    ylim=[0, 80], figsz=(4, 3))

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.show()

In [ ]:
with plt.style.context(['ieee', 'notebook', 'no-latex']):
    for k1 in files.keys():

        ers = [cond_ers[k1][k2].mean(-1).cpu().numpy() for k2 in files[k1].keys()]

        species_fig, species_ax = plot_confidence_intervals(*ers, labels=list(files[k1].keys()),
                y_label='Average Uncertainty Reduction',

                ylim=[30, 60], alpha=0.05)

        species_ax.spines['top'].set_visible(False)
        species_ax.spines['right'].set_visible(False)

        plt.show()